# DEX vs CEX comparison

Demonstrates AMM pool math and CEX–DEX spread analysis using synthetic fixtures (no network required).

In [ ]:
from datetime import UTC, datetime

import polars as pl

from app.market_data.amm_pool import AmmPool
from app.market_data.amm_types import SwapDirection
from app.models.domain import AmmPoolSnapshot
from app.services.arbitrage_scanner import scan_arbitrage_opportunities

cex_mids = [2800, 2900, 3000, 3100, 3200]
amm_spot = 3000.0
pool = AmmPool(1000.0, amm_spot * 1000.0, fee_bps=30.0)
snapshot = AmmPoolSnapshot(
    pool_address="0xpool",
    base_reserve=pool.base_reserve,
    quote_reserve=pool.quote_reserve,
    spot_price=pool.spot_price(),
    timestamp=datetime.now(UTC),
)

rows = []
for cex_mid in cex_mids:
  spread_bps = abs(amm_spot - cex_mid) / cex_mid * 10_000
  opps = scan_arbitrage_opportunities(
      cex_mid=cex_mid,
      pool_snapshot=snapshot,
      trial_trade_size=0.5,
      cex_taker_fee_bps=10,
      amm_fee_bps=30,
      gas_limit=200_000,
      gas_price_gwei=30,
      eth_price_usd=cex_mid,
      min_edge_bps=-10_000,
  )
  net_bps = opps[0].net_edge_bps if opps else 0.0
  direction = opps[0].direction.value if opps else "none"
  rows.append({"cex_mid": cex_mid, "spread_bps": spread_bps, "net_edge_bps": net_bps, "direction": direction})

df = pl.DataFrame(rows)
df

In [ ]:
trade_sizes = [0.1, 0.5, 1.0, 2.0, 5.0]
slippage_rows = [
    {
        "trade_size_eth": size,
        "slippage_bps": pool.slippage_bps(size, SwapDirection.BASE_TO_QUOTE),
    }
    for size in trade_sizes
]
pl.DataFrame(slippage_rows)